<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/graph_rag/llama_index_autograft_integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Entity Deduplication at Ingestion with AutoGraft

GraphRAG pipelines extract entities from documents and write them into a graph store. The same real-world entity is usually spelled differently across documents: `Apple Inc.`, `Apple`, `Apple Incorporated`. Naively inserting every mention fragments the knowledge graph, so queries miss nodes and analytics get skewed. Fixing duplicates after ingestion is expensive and error-prone.

This notebook shows how to deduplicate entities **at ingestion** with [AutoGraft](https://github.com/jules-gd-dev/autograft-lib), a 4-layer entity resolution middleware:

1. **Layer 1 (Deterministic)**: exact string and alias matching (0 tokens)
2. **Layer 1.5 (Lexical)**: suffix-strip and acronym matching (0 tokens)
3. **Layer 2 (Semantic)**: vector cosine similarity (0 tokens)
4. **Layer 3 (LLM Arbiter)**: only for genuinely ambiguous cases (costs tokens)

The demo runs entirely locally: no Neo4j, no API key, no LLM call. Embeddings are pre-computed and deterministic, so the notebook is reproducible in seconds.


In [ ]:
!pip install -q autograft numpy rapidfuzz pydantic


In [ ]:
import zlib

import numpy as np

from autograft.core.resolver import resolve_entity
from autograft.layers.semantic import cosine_similarity
from autograft.models.entities import Entity, ExistingNode

DIM = 384  # all-MiniLM-L6-v2 embedding size, fully deterministic, no model needed


def _unit(seed: str) -> np.ndarray:
    rng = np.random.default_rng(zlib.crc32(seed.encode()))
    vec = rng.standard_normal(DIM)
    return vec / float(np.linalg.norm(vec))


def _blend(parts: list[tuple[np.ndarray, float]]) -> list[float]:
    vec = sum(w * v for v, w in parts)
    if not parts:
        return [0.0] * DIM
    return [float(x) for x in vec / float(np.linalg.norm(vec))]


_APPLE = _unit("concept:apple")
_COMPANY = _unit("concept:company")
_LANGUAGE = _unit("concept:language")
_SNAKE = _unit("concept:snake")
_ANIMAL = _unit("concept:animal")

APPLE_INC_EMB = _blend([(_APPLE, 1.0), (_COMPANY, 0.6)])
APPLE_EMB = _blend([(_APPLE, 1.0), (_COMPANY, 0.3)])
PY_LANG_EMB = _blend([(_LANGUAGE, 1.0), (_SNAKE, 0.5)])
PY_ANIMAL_EMB = _blend([(_SNAKE, 1.0), (_ANIMAL, 0.7)])


### 2. Preparing the Corpus

`graph` holds the nodes already in the graph store. `extracted` is what an LLM extractor would output from a batch of documents: three surface forms of the same company, plus a `Python` mention and a `Python` homonym (the animal). A naive insert would create 7 nodes. AutoGraft resolves each mention against the graph before it is written.


In [ ]:
graph = [
    ExistingNode(node_id="n-apple", canonical_name="Apple Inc.", type="Organization", embedding=APPLE_INC_EMB),
    ExistingNode(node_id="n-python-lang", canonical_name="Python", type="ProgrammingLanguage", embedding=PY_LANG_EMB),
]

extracted = [
    Entity(canonical_name="Apple Inc.", type="Organization", embedding=APPLE_INC_EMB),
    Entity(canonical_name="Apple", type="Organization", embedding=APPLE_EMB),
    Entity(canonical_name="Apple Incorporated", type="Organization", embedding=APPLE_INC_EMB),
    Entity(canonical_name="Python", type="ProgrammingLanguage", embedding=PY_LANG_EMB),
    Entity(canonical_name="Python", type="Animal", embedding=PY_ANIMAL_EMB),
]


### 3. Resolving Every Mention

`resolve_entity(mention, db_client=graph)` runs the 4-layer cascade and returns the winning layer, the similarity score and the tokens consumed. It accepts a plain list of nodes, so it works without a running graph database.


In [ ]:
total_tokens = 0
print(f"Existing graph: {len(graph)} clean nodes\n")
for i, entity in enumerate(extracted, start=1):
    result = resolve_entity(entity, db_client=graph)
    total_tokens += result.tokens_used
    matched = next((n for n in graph if n.node_id == result.matched_node_id), None)
    semantic = (
        cosine_similarity(entity.embedding or [], matched.embedding or [])
        if matched and entity.embedding and matched.embedding
        else 0.0
    )
    if result.is_match:
        layer = f"{result.layer} (score {result.score:.1f})"
        outcome = f"merged into {result.matched_node_id}"
    else:
        layer = "declined"
        outcome = "new node (type gate: no same-type candidate)"
    sem = f"cos {semantic:.2f}" if semantic else "cos --"
    print(f"{i}. {entity.canonical_name:20s} ({entity.type:20s}) -> {layer:24s} {sem}  tokens {result.tokens_used}  {outcome}")

print(f"\nTotal LLM tokens used: {total_tokens} (Layer 3 never fired)")


### 4. What Happened, Layer by Layer

| # | Mention | Layer that resolved it |
|---|---------|------------------------|
| 1 | `Apple Inc.` | Layer 1 deterministic (score 100.0) |
| 2 | `Apple` | Layer 1.5 lexical, suffix-strip `Apple Inc.` -> `apple` (score 100.0) |
| 3 | `Apple Incorporated` | Layer 2 semantic, cosine 1.0 (the suffix list does not cover `Incorporated`, so the semantic layer catches it) |
| 4 | `Python` (language) | Layer 1 deterministic (score 100.0) |
| 5 | `Python` (animal) | Declined: the type gate keeps it out of `ProgrammingLanguage`, and its embedding (cos 0.43 < 0.75) would make Layer 2 decline too |

**Before**: a naive insert leaves 7 nodes. **After**: 3 clean nodes.
**Cost**: 0 LLM calls, 0 tokens across the whole run. Layer 3 (LLM arbiter) only activates on genuinely ambiguous `semantic_uncertain` results, so it never fired here.


### 5. Wiring It Into a LlamaIndex Pipeline

AutoGraft ships a drop-in middleware for LlamaIndex. Wrap your `Neo4jPropertyGraphStore` in one line and every `upsert_nodes` call is deduplicated locally before it reaches the database:

```python
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore
from autograft.integrations import AutoGraftLlamaIndexMiddleware

store = Neo4jPropertyGraphStore(username="neo4j", password="...", url="bolt://localhost:7687")
store = AutoGraftLlamaIndexMiddleware(store)

index = PropertyGraphIndex.from_documents(documents, property_graph_store=store)
```

No changes to your extraction pipeline. On a real 500-document run, AutoGraft resolved 99.8% of entity mentions locally: 7 LLM calls and 1,566 tokens instead of 3,000 calls and 661,714 tokens (see the [benchmark](https://github.com/jules-gd-dev/autograft-lib/blob/master/BENCHMARK.md)). The honest caveat: a few hard duplicates (~18 of 63 identities on the 500-doc corpus) still need an `alias_map` or an LLM arbiter call to be merged.
